# Qwen Nutrition · A100 40/80GB
Run top to bottom. Start with **SMOKE=True**, then use **False** for full training. Each completed dataset is cached independently on Drive. If one fails, rerun **only its cell**, then continue.

Image + optional text → mass, kcal, carbs, protein, fat. Grouped **72/8/20** train/validation/test split. Five-stage MLP → JSON workflow retained. Full A100 training, live MetaFood download, and phone accuracy are not certified here.


In [ ]:
#@title Connect Drive and check GPU
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,json,hashlib,subprocess,shutil
import torch
import builtins,functools
print=functools.partial(builtins.print,flush=True)
assert torch.cuda.is_available(), 'Choose an A100 GPU runtime.'
print(torch.cuda.get_device_name(0), 'VRAM GiB:', torch.cuda.get_device_properties(0).total_memory/2**30)
assert 'A100' in torch.cuda.get_device_name(0) and torch.cuda.get_device_properties(0).total_memory>35*2**30, 'This configuration requires an A100 with at least 40GB-class VRAM.'
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision('high')
os.environ['TOKENIZERS_PARALLELISM']='false'
IMPL=Path('/content/qwen_nutrition_impl');IMPL.mkdir(exist_ok=True)
sys.path.insert(0,str(IMPL))
META_ROOT=Path('/content/drive/MyDrive/MetaFood3D');META_ROOT.mkdir(parents=True,exist_ok=True)

Mounted at /content/drive
NVIDIA A100-SXM4-40GB VRAM GiB: 39.4935302734375


In [ ]:
#@title Install notebook code and verified nutrition table
import io,zipfile,base64
_bundle=base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(_bundle)) as z:
    for name in ['progress.py', 'data.py', 'meta.py', 'download_meta.py', 'train.py', 'deploy.py', 'cache.py']:
        (IMPL/name).write_bytes(z.read(name))
    # The MetaFood3D nutrition table is not redistributed. To use MetaFood3D,
    # obtain the table from the dataset authors and place it at META_PROVIDED.
    META_PROVIDED=META_ROOT/'_MetaFood3D_new_complete_dataset_nutrition_v2.xlsx'
del _bundle
import importlib
importlib.invalidate_caches()
print('Notebook code ready; nutrition table verified.')

In [ ]:
#@title Install dependencies (bounded; prints live output)
import time,threading

def log(message): print(time.strftime('[%H:%M:%S] ')+str(message),flush=True)
def run_command(args,timeout=300,check=True,**kwargs):
 """Stream child output, print liveness, and kill only this command's process group on timeout."""
 import subprocess,os,signal,queue
 q=queue.Queue();started=time.monotonic();last=started
 command=[str(x) for x in args];log('COMMAND: '+' '.join(command))
 proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
                       bufsize=1,start_new_session=True,**kwargs)
 def read():
  try:
   for line in proc.stdout:q.put(line.rstrip())
  finally:q.put(None)
 reader=threading.Thread(target=read,daemon=True);reader.start();ended=False
 def stop():
  if proc.poll() is None:
   os.killpg(proc.pid,signal.SIGTERM)
   try:proc.wait(timeout=5)
   except subprocess.TimeoutExpired:os.killpg(proc.pid,signal.SIGKILL);proc.wait(timeout=5)
 try:
  while not ended or proc.poll() is None:
   try:
    line=q.get(timeout=.2)
    if line is None:ended=True
    else:print(line,flush=True)
   except queue.Empty:pass
   if time.monotonic()-last>=20:
    log(f'Command still running: {time.monotonic()-started:.0f}s / {timeout}s limit');last=time.monotonic()
   if time.monotonic()-started>timeout:
    stop();raise subprocess.TimeoutExpired(command,timeout)
  result=proc.wait()
  if check and result:raise subprocess.CalledProcessError(result,command)
  log(f'Command finished: exit {result}, {time.monotonic()-started:.1f}s');return result
 except BaseException:
  stop();raise
 finally:
  reader.join(timeout=1);proc.stdout.close()

run_command([sys.executable,'-m','pip','install','--disable-pip-version-check','--retries','1','--timeout','30',
             'transformers==5.3.0','huggingface-hub>=1.3,<2','pillow','pandas','openpyxl','safetensors',
             'sentencepiece','playwright','tqdm'],timeout=600)
if shutil.which('ffmpeg') is None:
    run_command(['apt-get','-qq','update'],timeout=180)
    run_command(['apt-get','-qq','install','-y','ffmpeg'],timeout=300)

[02:27:16] COMMAND: /usr/bin/python3 -m pip install --disable-pip-version-check --retries 1 --timeout 30 transformers==5.3.0 huggingface-hub>=1.3,<2 pillow pandas openpyxl safetensors sentencepiece playwright tqdm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successfully uninstalled tokenizers-0.23.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
[02:27:33] Command finished: exit 0, 16.5s


In [ ]:
#@title Run settings
SMOKE = False             # First run: True. Full training: False.
DIRECT_JSON_ONLY = False  # Separate SFT-only ablation.
DOWNLOAD_META = True
INSTALL_FAST_KERNELS = False

In [ ]:
#@title Optional acceleration: disabled by default; never compiles CUDA
if INSTALL_FAST_KERNELS:
    for packages in [['flash-linear-attention==0.5.2','fla-core==0.5.2','einops'],['causal-conv1d']]:
        try:
            code=run_command([sys.executable,'-m','pip','install','--only-binary=:all:','--no-deps',
                 '--disable-pip-version-check','--retries','0','--timeout','20',*packages],timeout=180,check=False)
            if code: print('Optional wheel unavailable: skipped. Continue to GPU preflight.')
        except subprocess.TimeoutExpired:
            print('Optional installation timed out and was stopped. Continue without it.')
else:
    print('Optional installation skipped. Already-installed kernels remain available.')

Optional installation skipped. Already-installed kernels remain available.


In [ ]:
#@title Initialize run and dataset settings
RUN_ID = 'qwen35_nutrition_v3' + ('_direct' if DIRECT_JSON_ONLY else '') + ('_smoke' if SMOKE else '')
SNAP_MODE = 'train'  # 'external' uses all SNAPMe only as an external benchmark
assert SNAP_MODE in ['train','external']
VRAM_GIB=torch.cuda.get_device_properties(0).total_memory/2**30
CFG=dict(seed=42,batch=1 if VRAM_GIB<70 else 2,accum=16 if VRAM_GIB<70 else 8,workers=2,max_length=1536,reason_prob=.35,
         text_layers=12,vision_layers=12,gradient_checkpointing=True,
         save_minutes=30,val_per_source=8 if SMOKE else 64,
         smoke=SMOKE,direct_json_only=DIRECT_JSON_ONLY)
CFG['drive']=str(Path('/content/drive/MyDrive/QwenNutrition')/RUN_ID)
CFG['local']=str(Path('/content/qwen_nutrition')/RUN_ID)
for k in ['drive','local']:Path(CFG[k]).mkdir(parents=True,exist_ok=True)
MODEL_ID='Qwen/Qwen3.5-2B'
REVISION='15852e8c16360a2fea060d615a32b45270f8a8fc'
DATA_ROOT=Path('/content/nutrition_data');DATA_ROOT.mkdir(exist_ok=True)
DRIVE_CACHE=Path('/content/drive/MyDrive/QwenNutrition/dataset_cache_v1')
LOCAL_CACHE=Path('/content/nutrition_cached_rgb')
for _module_name in ['progress','data','meta','download_meta','train','deploy','cache']:
    if _module_name in sys.modules: importlib.reload(sys.modules[_module_name])
from cache import dataset
from progress import log,stage,run_command,progress
from meta import metafood
from data import nutrition5k,snapme,clean,split,K,sha
from meta import metafood
from train import *
seed(CFG['seed'])

print('Memory configuration:',CFG['batch'],'microbatch x',CFG['accum'],'accumulation steps; run SMOKE through a real optimizer step before full training.')

REFRESH_DATASETS = []  # Optional: 'nutrition5k', 'snapme', 'metafood3d'
SNAP_ARCHIVE = None    # Optional local tar.gz instead of the official download
print('Cache directory:',DRIVE_CACHE)

META_XLSX=META_PROVIDED
META_SHEET='Sheet1'
META_MAPPING={'object_id':'Food_Type','total_mass_g':'Weight (g)','calories_kcal':'Energy (Kcal)',
              'carbs_g':'Carbs (g)','protein_g':'Protein (g)','fat_g':'Fat (g)'}
META_BASIS='per_object';META_ENERGY_UNIT='kcal'
META_RGB_PATTERN=r'RGBD_videos/'
META_ID_COMPONENT=None

Memory configuration: 1 microbatch x 16 accumulation steps; run SMOKE through a real optimizer step before full training.
Cache directory: /content/drive/MyDrive/QwenNutrition/dataset_cache_v1


### 1 · Nutrition5k
Completed caches bypass the downloader. Existing local files are reused during the first cache build.

In [ ]:
n5,a5=dataset('nutrition5k',
    lambda:nutrition5k(DATA_ROOT/'nutrition5k',fill_side=True),
    DRIVE_CACHE,LOCAL_CACHE,recipe={'adapter':3,'side_frames':True},
    refresh='nutrition5k' in REFRESH_DATASETS)

[02:27:34] START Dataset cache: nutrition5k
[02:27:35] nutrition5k: COMPLETE Drive cache found; no network listing or download will run
[02:27:35] nutrition5k restore/check local RGB cache: starting 4959 images


[02:27:56] nutrition5k restore/check local RGB cache: 6/4959 | elapsed 20s
[02:28:16] nutrition5k restore/check local RGB cache: 33/4959 | elapsed 41s
[02:28:37] nutrition5k restore/check local RGB cache: 59/4959 | elapsed 61s
[02:28:57] nutrition5k restore/check local RGB cache: 83/4959 | elapsed 82s
[02:29:18] nutrition5k restore/check local RGB cache: 110/4959 | elapsed 103s
[02:29:38] nutrition5k restore/check local RGB cache: 136/4959 | elapsed 123s
[02:29:58] nutrition5k restore/check local RGB cache: 161/4959 | elapsed 143s
[02:30:19] nutrition5k restore/check local RGB cache: 187/4959 | elapsed 163s
[02:30:39] nutrition5k restore/check local RGB cache: 214/4959 | elapsed 183s
[02:30:59] nutrition5k restore/check local RGB cache: 239/4959 | elapsed 204s
[02:31:19] nutrition5k restore/check local RGB cache: 263/4959 | elapsed 224s
[02:31:40] nutrition5k restore/check local RGB cache: 288/4959 | elapsed 244s
[02:32:00] nutrition5k restore/check local RGB cache: 312/4959 | elapsed 

### 2 · SNAPMe

In [ ]:
%run -i /content/apply_snapme_fix.py

SNAPMe adapter fixed. Other dataset caches are unchanged.
[03:35:28] START Dataset cache: snapme
[03:35:29] snapme: COMPLETE Drive cache found; no network listing or download will run
[03:35:29] snapme restore/check local RGB cache: starting 1459 images


[03:35:49] snapme restore/check local RGB cache: 7/1459 | elapsed 20s
[03:36:09] snapme restore/check local RGB cache: 28/1459 | elapsed 40s
[03:36:30] snapme restore/check local RGB cache: 51/1459 | elapsed 61s
[03:36:51] snapme restore/check local RGB cache: 75/1459 | elapsed 82s
[03:37:12] snapme restore/check local RGB cache: 99/1459 | elapsed 103s
[03:37:32] snapme restore/check local RGB cache: 122/1459 | elapsed 123s
[03:37:53] snapme restore/check local RGB cache: 145/1459 | elapsed 144s
[03:38:14] snapme restore/check local RGB cache: 168/1459 | elapsed 165s
[03:38:34] snapme restore/check local RGB cache: 182/1459 | elapsed 185s
[03:38:55] snapme restore/check local RGB cache: 206/1459 | elapsed 206s
[03:39:15] snapme restore/check local RGB cache: 229/1459 | elapsed 226s
[03:39:35] snapme restore/check local RGB cache: 252/1459 | elapsed 246s
[03:39:56] snapme restore/check local RGB cache: 275/1459 | elapsed 266s
[03:40:16] snapme restore/check local RGB cache: 298/1459 | e

In [ ]:
sn,asn=dataset('snapme',
    lambda:snapme(DATA_ROOT/'snapme',local_archive=SNAP_ARCHIVE),
    DRIVE_CACHE,LOCAL_CACHE,recipe={'adapter':3,'revision':'ce09cee4a1e0c5b05f35abe5a8a058e686fefa74'},
    refresh='snapme' in REFRESH_DATASETS)

[10:15:02] START Dataset cache: snapme
[10:15:02] FAILED Dataset cache: snapme: RuntimeError: snapme: cache recipe changed. Add only this source to REFRESH_DATASETS to rebuild it.


RuntimeError: snapme: cache recipe changed. Add only this source to REFRESH_DATASETS to rebuild it.

### 3 · MetaFood3D
Uses the corrected checkbox selector and your embedded nutrition table. Enter the password only in the hidden prompt. Set `DOWNLOAD_META=False` if the RGB archive is already placed in `MyDrive/MetaFood3D`.

In [ ]:
DOWNLOAD_META = False
mt = []
amt = {"status": "excluded_by_user", "loaded": 0}
print("MetaFood3D skipped. Using Nutrition5k + SNAPMe.")

In [ ]:
meta_complete=(DRIVE_CACHE/'metafood3d'/'COMPLETE.json').exists()
if (not meta_complete or 'metafood3d' in REFRESH_DATASETS) and DOWNLOAD_META:
    _raw_marker=META_ROOT/'download_complete.json'
    try:
        _raw_files=json.loads(_raw_marker.read_text())
        _raw_complete=bool(_raw_files) and all((META_ROOT/x['name']).is_file() and (META_ROOT/x['name']).stat().st_size==x['size'] for x in _raw_files)
    except (OSError,ValueError,KeyError,TypeError): _raw_complete=False
    if not _raw_complete:
        run_command([sys.executable,'-m','playwright','install','--with-deps','chromium'],timeout=600)
    from download_meta import download_meta
    await download_meta(META_ROOT)
mt,amt=dataset('metafood3d',
    lambda:metafood(META_ROOT,META_XLSX,META_SHEET,META_MAPPING,META_BASIS,META_ENERGY_UNIT,
                   DATA_ROOT/'meta_rgb',views=2,rgb_pattern=META_RGB_PATTERN,id_component=META_ID_COMPONENT),
    DRIVE_CACHE,LOCAL_CACHE,
    recipe={'adapter':3,'xlsx':sha(META_XLSX),'mapping':META_MAPPING,'basis':META_BASIS,
            'energy':META_ENERGY_UNIT,'views':2,'rgb_pattern':META_RGB_PATTERN,'id_component':META_ID_COMPONENT},
    refresh='metafood3d' in REFRESH_DATASETS)

[08:35:59] COMMAND: /usr/bin/python3 -m playwright install --with-deps chromium
Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,890 kB]
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,545 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,047 kB]
Get:12 https://ppa.l

[08:36:36] Waiting/downloading; bytes in new Drive download files: 0
[08:36:56] Waiting/downloading; bytes in new Drive download files: 299122688
[08:37:16] Waiting/downloading; bytes in new Drive download files: 449527808
[08:37:36] Waiting/downloading; bytes in new Drive download files: 633372672
[08:37:56] Waiting/downloading; bytes in new Drive download files: 819675136
[08:38:16] Waiting/downloading; bytes in new Drive download files: 1016446976
[08:38:36] Waiting/downloading; bytes in new Drive download files: 1161117696
[08:38:56] Waiting/downloading; bytes in new Drive download files: 1323253760
[08:39:16] Waiting/downloading; bytes in new Drive download files: 1484931072
[08:39:36] Waiting/downloading; bytes in new Drive download files: 1656750080
[08:39:56] Waiting/downloading; bytes in new Drive download files: 1832075264
[08:40:16] Waiting/downloading; bytes in new Drive download files: 1991245824
[08:40:36] Waiting/downloading; bytes in new Drive download files: 2158297088

RuntimeError: Unexpected downloaded file type: _MetaFood3D_new_RGBD_videos.tar.gz

### 4 · Validate and split

In [ ]:
rows,rejected=clean(n5+sn+mt)
log('5/6 Grouped train/validation/test split')
external=[r for r in rows if r['source']=='snapme'] if SNAP_MODE=='external' else []
rows=[r for r in rows if not (SNAP_MODE=='external' and r['source']=='snapme')]
rows,counts=split(rows,CFG['seed'])
for r in external:r['split']='external'
print('SPLITS:',json.dumps(counts,indent=2))
print('SOURCE AUDIT:',json.dumps({'nutrition5k':a5,'snapme':asn,'metafood3d':amt},indent=2))
print('Rejected/deduplicated image audit:',len(rejected))
assert len({r['id'] for r in rows})==len(rows), 'Duplicate sample IDs'
print('Preparation-note coverage:',sum(bool(r.get('verified_notes')) for r in rows))
TRAIN=[r for r in rows if r['split']=='train'];VAL=[r for r in rows if r['split']=='val'];TEST=[r for r in rows if r['split']=='test']
assert TRAIN and VAL and TEST

[03:58:42] START clean
[03:58:42] Decode images + pixel hashes: starting 6418 images


[03:59:02] Decode images + pixel hashes: 1298/6418 | elapsed 20s
[03:59:22] Decode images + pixel hashes: 2766/6418 | elapsed 40s
[03:59:42] Decode images + pixel hashes: 3713/6418 | elapsed 60s
[04:00:02] Decode images + pixel hashes: 4193/6418 | elapsed 80s
[04:00:22] Decode images + pixel hashes: 4670/6418 | elapsed 100s
[04:00:42] Decode images + pixel hashes: 5288/6418 | elapsed 120s
[04:01:02] Decode images + pixel hashes: 5819/6418 | elapsed 140s
[04:01:22] Decode images + pixel hashes: 6271/6418 | elapsed 161s
[04:01:32] Decode images + pixel hashes: completed 6418 images in 170.8s
[04:01:32] Image validation: 6415 retained / 6418 inputs; 0 rejected, 0 cross-group duplicate hashes
[04:01:32] DONE clean | 170.8s
[04:01:32] 5/6 Grouped train/validation/test split
[04:01:32] START split
[04:01:32] Split nutrition5k: 4959 rows, 122 independent groups
[04:01:32] Group selection (target 992): starting 122 groups


[04:01:33] Group selection (target 992): completed 122 groups in 0.2s
[04:01:33] Group selection (target 397): starting 97 groups


[04:01:33] Group selection (target 397): completed 97 groups in 0.0s
[04:01:33] Split nutrition5k: {'train': 3570, 'test': 992, 'val': 397}
[04:01:33] Split snapme: 1456 rows, 95 independent groups
[04:01:33] Group selection (target 291): starting 95 groups


[04:01:33] Group selection (target 291): completed 95 groups in 0.0s
[04:01:33] Group selection (target 116): starting 76 groups


[04:01:33] Group selection (target 116): completed 76 groups in 0.0s
[04:01:33] Split snapme: {'val': 116, 'test': 291, 'train': 1049}
[04:01:33] DONE split | 0.3s
SPLITS: {
  "nutrition5k": {
    "train": 3570,
    "test": 992,
    "val": 397
  },
  "snapme": {
    "val": 116,
    "test": 291,
    "train": 1049
  }
}
SOURCE AUDIT: {
  "nutrition5k": {
    "cafe1": {
      "rejected": {},
      "sha256": "881df31f27c2343d0ce219964527e0796f6ddadd60701ce6840fa801e8c2a960"
    },
    "cafe2": {
      "rejected": {},
      "sha256": "ef9f800d7b9c2e7e9a9e7cda5cbb4cfe51c760a8c1af4c05786ccf051bf5d77d"
    },
    "loaded": 4959,
    "missing_rgb": 47
  },
  "snapme": {
    "incomplete_photo": 0,
    "excluded_packaged_or_mixed_photo": 203,
    "invalid_filename_photo": 19,
    "loaded": 1459,
    "missing_images": 14,
    "missing_filename_examples": [
      "75a3490443cd98c9d3ad0a4d17f5.jpeg",
      "be1bd8a54d618914ad87758b8bb4.jpeg",
      "7854c3f24885855f177b892bf6ce.jpeg",
      "27ac723

In [ ]:
#@title Save reproducible experiment manifest
with stage('6/6 Build signature and save manifests to Drive'):
    # The content signature is independent of the ephemeral Colab image paths.
    identity=[{k:r[k] for k in ['id','source','group','y','image_sha256','split','ingredients','reason_ok','weight','basis']} for r in sorted(rows+external,key=lambda x:x['id'])]
    for item,r in zip(identity,sorted(rows+external,key=lambda x:x['id'])):item['verified_notes']=r.get('verified_notes',[])
    settings={k:v for k,v in CFG.items() if k not in ['drive','local','workers','save_minutes']}
    spec=dict(model=MODEL_ID,revision=REVISION,settings=settings,snap_mode=SNAP_MODE,rows=identity,
              code={p.name:sha(p) for p in sorted(IMPL.glob('*.py'))})
    SIGNATURE=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
    meta_path=Path(CFG['drive'])/'experiment.json'
    if meta_path.exists():
        assert json.loads(meta_path.read_text())['signature']==SIGNATURE, 'Run configuration/data/code changed. Choose a new RUN_ID.'
    else:meta_path.write_text(json.dumps(dict(signature=SIGNATURE,spec=spec),indent=2))
    Path(CFG['drive'],'manifest.json').write_text(json.dumps(rows+external,ensure_ascii=False,indent=2))
    Path(CFG['drive'],'data_audit.json').write_text(json.dumps(dict(nutrition5k=a5,snapme=asn,metafood3d=amt,rejected=rejected,counts=counts),indent=2))
    Path(CFG['drive'],'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    if SMOKE:
        TRAIN=val_subset(TRAIN,12);VAL=val_subset(VAL,8);TEST=val_subset(TEST,8)
    print('Used:',len(TRAIN),'train,',len(VAL),'validation,',len(TEST),'test; signature',SIGNATURE[:12])

[04:17:31] START 6/6 Build signature and save manifests to Drive
Used: 4619 train, 513 validation, 1283 test; signature 45e350f749fb
[04:17:33] DONE 6/6 Build signature and save manifests to Drive | 1.9s


### 5 · Load model and check gradients

In [ ]:
with stage('Load processor and model'):
    from transformers import AutoProcessor,Qwen3_5ForConditionalGeneration
    PROCESSOR=AutoProcessor.from_pretrained(MODEL_ID,revision=REVISION,min_pixels=128*128,max_pixels=512*512)
    PROCESSOR.tokenizer.padding_side='right'
    BASE=Qwen3_5ForConditionalGeneration.from_pretrained(MODEL_ID,revision=REVISION,
        torch_dtype=torch.float32,attn_implementation='sdpa').cuda()
    BASE.config.use_cache=False
    SCALE=np.maximum(np.mean([r['y'] for r in TRAIN],axis=0),[1,1,1,1,1]).tolist()
    NET=Model(BASE,SCALE).cuda()
    CP=Checkpoints(Path(CFG['drive'])/'checkpoints',Path(CFG['local'])/'checkpoint_tmp',SIGNATURE)
    print('Base FP32 parameters GB:',sum(p.numel()*p.element_size() for p in BASE.parameters())/1e9)

In [ ]:
with stage('GPU preflight (backward only; SMOKE also checks optimizer allocation)'):
    torch.cuda.reset_peak_memory_stats()
    probe=move(Collator(PROCESSOR,sft=True,reason_prob=0,max_length=CFG['max_length'])(TRAIN[:1]))
    for st in STAGES[:3]:
        NET.zero_grad(set_to_none=True);stage_setup(NET,st,CFG)
        with amp():loss=NET.loss(probe,st['reg'],st['ce'])
        assert torch.isfinite(loss)
        loss.backward()
        has=lambda module:any(p.grad is not None and bool(p.grad.detach().abs().sum()>0) for p in module.parameters())
        assert has(NET.head)==(st['name']!='03_frozen_head')
        if st['name']=='02_head':assert not has(NET.base)
        else:
            assert has(NET.base.model.visual), 'No vision gradient'
            assert has(NET.base.model.language_model), 'No language gradient'
        print('PASS',st['name'],float(loss.detach()))
    NET.zero_grad(set_to_none=True);del probe,loss
    print('Preflight peak allocated GiB:',torch.cuda.max_memory_allocated()/2**30)
    torch.cuda.empty_cache()

[04:02:31] START GPU preflight (backward only; SMOKE also checks optimizer allocation)
02_head trainable: 1,119,493
PASS 02_head 0.039844810962677
03_frozen_head trainable: 862,962,592
PASS 03_frozen_head 0.039844810962677
04_joint_bridge trainable: 864,082,085
PASS 04_joint_bridge 1.0153813362121582
Preflight peak allocated GiB: 11.540902137756348
[04:02:39] DONE GPU preflight (backward only; SMOKE also checks optimizer allocation) | 7.8s


### 6 · Train
Checkpoints include optimizer/RNG state and resume automatically for the same run. The new `v3` run ID preserves older experiments. A 20-second heartbeat means “still running,” not necessarily “network transferring.”

In [ ]:
import json
from pathlib import Path

root = Path(CFG["drive"]) / "checkpoints"
matching = []

for marker in sorted(root.glob("ckpt_*/COMPLETE.json"), reverse=True):
    try:
        info = json.loads(marker.read_text())
        if (info.get("signature") == SIGNATURE
                and (marker.parent / "state.pt").is_file()):
            matching.append((marker.parent, info))
    except (OSError, ValueError):
        continue

assert matching, "No matching checkpoint found. Stopped to prevent restarting."

print("Latest checkpoint:", matching[0][0])
print("Saved progress:", matching[0][1]["progress"])

EVAL_DIR = Path(CFG["drive"]) / "evaluation"
EVAL_DIR.mkdir(exist_ok=True)
fit(NET, PROCESSOR, TRAIN, VAL, CFG, CP)

Latest checkpoint: /content/drive/MyDrive/QwenNutrition/qwen35_nutrition_v3/checkpoints/ckpt_1791567257414789818
Saved progress: {'stage': 3, 'epoch': 0, 'next_batch': 3632, 'step': 227}
[04:17:51] START fit
[04:17:52] START Restore checkpoint from Drive
[04:26:00] DONE Restore checkpoint from Drive | 488.0s
[04:26:00] Checksum: resume.pt


05_json trainable: 862,962,592
05_json trainable: 862,962,592
[04:27:04] 05_json epoch 1/1: starting 4619 batchs


[04:27:24] 05_json epoch 1/1: 865/4619 | elapsed 20s
[04:27:44] 05_json epoch 1/1: 1784/4619 | elapsed 40s
[04:28:04] 05_json epoch 1/1: 2674/4619 | elapsed 60s
[04:28:24] 05_json epoch 1/1: 3559/4619 | elapsed 80s
[04:28:44] 05_json epoch 1/1: 3642/4619 | elapsed 100s
[04:29:06] 05_json epoch 1/1: 3653/4619 | elapsed 122s
[04:29:27] 05_json epoch 1/1: 3664/4619 | elapsed 143s
[04:29:48] 05_json epoch 1/1: 3675/4619 | elapsed 164s
05_json epoch 1 batch 3680 / 4619 loss 0.5949


/content/qwen_nutrition_impl/train.py:287: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  if step%10==0:print(s['name'],'epoch',epoch+1,'batch',bi+1,'/',n,'loss',round(float(loss),4),flush=True)


[04:30:08] 05_json epoch 1/1: 3686/4619 | elapsed 185s
[04:30:29] 05_json epoch 1/1: 3697/4619 | elapsed 205s
[04:30:50] 05_json epoch 1/1: 3708/4619 | elapsed 227s
[04:31:11] 05_json epoch 1/1: 3719/4619 | elapsed 248s
[04:31:33] 05_json epoch 1/1: 3730/4619 | elapsed 269s
[04:31:54] 05_json epoch 1/1: 3741/4619 | elapsed 290s
[04:32:16] 05_json epoch 1/1: 3752/4619 | elapsed 312s
[04:32:37] 05_json epoch 1/1: 3763/4619 | elapsed 333s
[04:32:58] 05_json epoch 1/1: 3774/4619 | elapsed 354s
[04:33:20] 05_json epoch 1/1: 3785/4619 | elapsed 376s
[04:33:41] 05_json epoch 1/1: 3796/4619 | elapsed 398s
[04:34:03] 05_json epoch 1/1: 3807/4619 | elapsed 419s
[04:34:24] 05_json epoch 1/1: 3818/4619 | elapsed 440s
[04:34:45] 05_json epoch 1/1: 3828/4619 | elapsed 461s
[04:35:06] 05_json epoch 1/1: 3839/4619 | elapsed 483s
05_json epoch 1 batch 3840 / 4619 loss 0.1991
[04:35:26] 05_json epoch 1/1: 3849/4619 | elapsed 503s
[04:35:47] 05_json epoch 1/1: 3860/4619 | elapsed 523s
[04:36:07] 05_json 

...

[05:08:34] Evaluation thinking=False, text=0: 128/128 | elapsed 336s
[05:08:34] Evaluation thinking=False, text=0: completed 128 photos in 336.2s
[05:08:34] DONE evaluate | 336.2s
[05:08:34] START Save best weights: 05_json
[05:08:57] DONE Save best weights: 05_json | 23.0s
[05:09:00] START Checkpoint: serialize local state
[05:09:43] DONE Checkpoint: serialize local state | 42.5s
[05:09:43] START Checkpoint: copy to Drive
[05:10:33] DONE Checkpoint: copy to Drive | 50.1s
[05:10:33] Checkpoint: verifying SHA256 on local and Drive copies
[05:10:33] Checksum: state.pt


[05:11:25] Checksum: state.pt


Checkpoint complete: {'stage': 3, 'epoch': 1, 'next_batch': 0, 'step': 289}
[05:12:48] START Load best weights: 05_json
[05:13:14] DONE Load best weights: 05_json | 26.0s
[05:13:14] START Checkpoint: serialize local state
[05:13:53] DONE Checkpoint: serialize local state | 39.0s
[05:13:53] START Checkpoint: copy to Drive
[05:15:16] DONE Checkpoint: copy to Drive | 83.0s
[05:15:16] Checkpoint: verifying SHA256 on local and Drive copies
[05:15:16] Checksum: state.pt


[05:16:04] Checksum: state.pt


Checkpoint complete: {'stage': 3, 'epoch': 1, 'next_batch': 0, 'step': 289, 'complete': True}
[05:17:27] DONE fit | 3575.8s


In [ ]:
# Original-model validation before any optimizer update. Cached to Drive.
EVAL_DIR=Path(CFG['drive'])/'evaluation';EVAL_DIR.mkdir(exist_ok=True)
if not list((Path(CFG['drive'])/'checkpoints').glob('ckpt_*/COMPLETE.json')):
    baseline_val=evaluate(BASE,PROCESSOR,val_subset(VAL,CFG['val_per_source']),EVAL_DIR/'baseline_val_no_think.jsonl')
    (EVAL_DIR/'baseline_val_metrics.json').write_text(json.dumps(baseline_val,indent=2))
fit(NET,PROCESSOR,TRAIN,VAL,CFG,CP)

NameError: name 'Path' is not defined

### 7 · Evaluate and export

In [ ]:
mode_scores={}
for thinking in [False,True]:
    vv=evaluate(NET.base,PROCESSOR,val_subset(VAL,CFG['val_per_source']),
                EVAL_DIR/f'final_val_think_{thinking}.jsonl',thinking=thinking)
    mode_scores[str(thinking)]=vv
THINKING=min([False,True],key=lambda x:mode_scores[str(x)]['all']['score'])
(EVAL_DIR/'mode_selection.json').write_text(json.dumps(dict(thinking=THINKING,validation=mode_scores),indent=2))
RESULTS={}
for variant in [0,1]:
    RESULTS[f'final_test_variant_{variant}']=evaluate(NET.base,PROCESSOR,TEST,
        EVAL_DIR/f'final_test_think_{THINKING}_variant_{variant}.jsonl',thinking=THINKING,variant=variant)
if external:
    RESULTS['external_snapme']=evaluate(NET.base,PROCESSOR,external,EVAL_DIR/'external_snapme.jsonl',thinking=THINKING)
(EVAL_DIR/'final_metrics.json').write_text(json.dumps(RESULTS,indent=2))
print(json.dumps(RESULTS,indent=2))

[05:18:07] START evaluate
[05:18:07] Evaluation thinking=False, text=0: starting 128 photos


[05:18:29] Evaluation thinking=False, text=0: 8/128 | elapsed 22s


[05:18:51] Evaluation thinking=False, text=0: 16/128 | elapsed 43s


[05:19:13] Evaluation thinking=False, text=0: 24/128 | elapsed 66s


eval 25 / 128


...

[07:36:52] Evaluation thinking=False, text=1: 1262/1283 | elapsed 3280s


[07:37:13] Evaluation thinking=False, text=1: 1270/1283 | elapsed 3301s


eval 1275 / 1283


[07:37:33] Evaluation thinking=False, text=1: 1278/1283 | elapsed 3321s


[07:37:46] Evaluation thinking=False, text=1: completed 1283 photos in 3333.9s
[07:37:46] DONE evaluate | 3333.9s
{
  "final_test_variant_0": {
    "all": {
      "n": 1273,
      "total_mass_g": {
        "mae": 55.47012538658209,
        "rmse": 329.4809158334331,
        "wape_pct": 27.989197527462206
      },
      "calories_kcal": {
        "mae": 69.31478009113906,
        "rmse": 157.31309938951665,
        "wape_pct": 31.007366319176196
      },
      "carbs_g": {
        "mae": 7.434371716641791,
        "rmse": 26.397330522891096,
        "wape_pct": 37.17877058875439
      },
      "protein_g": {
        "mae": 4.7978979901366845,
        "rmse": 9.46248246146725,
        "wape_pct": 34.23405829321322
      },
      "fat_g": {
        "mae": 4.306486588146112,
        "rmse": 7.809553095908797,
        "wape_pct": 42.16391661644933
      },
      "total": 1283,
      "invalid_json": 10,
      "valid_rate": 0.9922057677318784,
      "score": 42.30889413713266
    },
    "nutr

In [ ]:
with stage('Export BF16 weights and processor'):
    EXPORT=Path(CFG['drive'])/'export_bf16';EXPORT.mkdir(exist_ok=True)
    assert NET.head is None, 'MLP must be removed before deployment export.'
    NET.base.to(torch.bfloat16)
    NET.base.save_pretrained(EXPORT,safe_serialization=True,max_shard_size='2GB')
    PROCESSOR.save_pretrained(EXPORT)
    (EXPORT/'nutrition_spec.json').write_text(json.dumps(dict(keys=K,signature=SIGNATURE,thinking=THINKING,
        model=MODEL_ID,revision=REVISION,max_deployment_bytes=2_000_000_000,results=RESULTS),indent=2))
    print('Exported BF16 GB:',sum(p.stat().st_size for p in EXPORT.glob('*.safetensors'))/1e9)
    # Replace with your own image and optional note. Empty text is supported.
    USER_IMAGE=TEST[0]['image']
    USER_NOTE=''
    raw,parsed=generate(NET.base,PROCESSOR,{'image':USER_IMAGE},USER_NOTE,THINKING)
    print('Valid final JSON:',dict(zip(K,parsed)) if parsed is not None else 'INVALID')

[07:38:31] START Export BF16 weights and processor


Exported BF16 GB: 4.426558168


Valid final JSON: {'total_mass_g': 39.0, 'calories_kcal': 220.1, 'carbs_g': 7.8, 'protein_g': 7.9, 'fat_g': 19.8}
[07:38:49] DONE Export BF16 weights and processor | 18.7s


In [ ]:
with stage('Original-model baseline on the same test set'):
    del NET,BASE
    import gc
    gc.collect();torch.cuda.empty_cache()
    ORIGINAL=Qwen3_5ForConditionalGeneration.from_pretrained(MODEL_ID,revision=REVISION,
        torch_dtype=torch.bfloat16,attn_implementation='sdpa').cuda().eval()
    for variant in [0,1]:
        baseline=evaluate(ORIGINAL,PROCESSOR,TEST,EVAL_DIR/f'baseline_test_think_{THINKING}_variant_{variant}.jsonl',thinking=THINKING,variant=variant)
        RESULTS[f'baseline_test_variant_{variant}']=baseline
    (EVAL_DIR/'comparison.json').write_text(json.dumps(RESULTS,indent=2))
    print(json.dumps(RESULTS,indent=2))
    del ORIGINAL;gc.collect();torch.cuda.empty_cache()

[07:38:49] START Original-model baseline on the same test set


[07:39:06] START evaluate
[07:39:06] Evaluation thinking=False, text=0: starting 1283 photos


[07:39:26] Evaluation thinking=False, text=0: 6/1283 | elapsed 20s


[07:39:49] Evaluation thinking=False, text=0: 13/1283 | elapsed 43s


[07:40:11] Evaluation thinking=False, text=0: 20/1283 | elapsed 65s


...

[08:37:31] Evaluation thinking=False, text=0: 1157/1283 | elapsed 3505s


[08:37:54] Evaluation thinking=False, text=0: 1164/1283 | elapsed 3527s


[08:38:16] Evaluation thinking=False, text=0: 1171/1283 | elapsed 3550s


eval 1175 / 1283


[08:38:38] Evaluation thinking=False, text=0: 1178/1283 | elapsed 3572s


In [ ]:
RUN_QUANTIZATION = not SMOKE
DEPLOYMENT_FILES=[]
if RUN_QUANTIZATION:
    from deploy import build_mobile
    DEPLOYMENT_FILES=build_mobile(EXPORT,CFG['drive'],CFG['local'])
    assert sum(p.stat().st_size for p in DEPLOYMENT_FILES)<=2_000_000_000
else:
    print('Quantization skipped for smoke run; no compliant mobile export claimed.')

### Notes
- Nutrition5k, SNAPMe, and complete unambiguous MetaFood3D rows only. OCR/ECUSTFD excluded. SNAPMe uses reported consumption and weight 0.30; descriptions may reveal more than a real user would provide.
- Reasoning is selected on validation. JSON is validated, not guaranteed by constrained decoding.
- Quantized **text + vision weights must total ≤2GB**. BF16 export is larger; phone testing is still required.
- Keep the same run settings to resume. To rebuild one source only, add its name to `REFRESH_DATASETS`.
